# Exploratory Data Analysis — RACE Reading Comprehension Dataset

**Course:** AL2002 Artificial Intelligence (Lab), Spring 2026  
**Dataset:** RACE (Lai et al., 2017) — English reading comprehension from Chinese school exams

This notebook explores the dataset structure, class balance, text length distributions, question-type breakdown, and lexical overlap statistics.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# Make src importable
_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from src.preprocessing import (
    OPTION_LETTERS, clean_text, tokenize, split_sentences,
    label_question_type, WH_LABELS
)
from src.demo_data import load_demo_dataframe

print('Imports OK')

## 1. Load Data

We attempt to load the real RACE CSVs. If they are not present, we fall back to the 8-sample synthetic demo dataset so this notebook always runs.

In [ ]:
DATA_DIR = os.path.join(_ROOT, 'data', 'raw')
USING_REAL_DATA = False

train_path = os.path.join(DATA_DIR, 'train.csv')
val_path   = os.path.join(DATA_DIR, 'val.csv')
test_path  = os.path.join(DATA_DIR, 'test.csv')

if os.path.exists(train_path):
    print('Loading real RACE CSVs ...')
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    USING_REAL_DATA = True
    print(f'Loaded {len(train_df)} train / {len(val_df)} val / {len(test_df)} test rows')
else:
    print('Real CSVs not found — using synthetic demo dataset (8 samples).')
    df = load_demo_dataframe()

print(f'\nTotal rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Basic Statistics

In [ ]:
n_unique_articles = df['article'].nunique()
n_questions = len(df)

print(f'Unique passages:  {n_unique_articles}')
print(f'Total questions:  {n_questions}')
print(f'Avg questions per passage: {n_questions / max(1, n_unique_articles):.1f}')
print(f'\nMissing values per column:')
print(df.isnull().sum())

## 3. Answer Letter Distribution

A well-balanced dataset should have roughly equal counts for A, B, C, D.

In [ ]:
answer_counts = df['answer'].str.strip().str.upper().value_counts().reindex(list(OPTION_LETTERS), fill_value=0)

fig = px.bar(
    x=answer_counts.index,
    y=answer_counts.values,
    labels={'x': 'Answer Letter', 'y': 'Count'},
    title='Answer Letter Distribution',
    color=answer_counts.index,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(showlegend=False)
fig.show()

print('Proportions:')
print((answer_counts / answer_counts.sum()).round(3))

## 4. Text Length Distributions

We compute word-level lengths for articles, questions, and answer options.

In [ ]:
df['article_len']  = df['article'].apply(lambda x: len(tokenize(x)))
df['question_len'] = df['question'].apply(lambda x: len(tokenize(x)))

option_lengths = []
for letter in OPTION_LETTERS:
    df[f'{letter}_len'] = df[letter].apply(lambda x: len(tokenize(x)))
    option_lengths.append(f'{letter}_len')

df['avg_option_len'] = df[option_lengths].mean(axis=1)

summary = pd.DataFrame({
    'Metric': ['Article (words)', 'Question (words)', 'Avg Option (words)'],
    'Mean':   [df['article_len'].mean(), df['question_len'].mean(), df['avg_option_len'].mean()],
    'Median': [df['article_len'].median(), df['question_len'].median(), df['avg_option_len'].median()],
    'Min':    [df['article_len'].min(), df['question_len'].min(), df['avg_option_len'].min()],
    'Max':    [df['article_len'].max(), df['question_len'].max(), df['avg_option_len'].max()],
}).round(1)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['article_len'], bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_title('Article Length (words)')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['question_len'], bins=20, color='#55A868', edgecolor='white')
axes[1].set_title('Question Length (words)')
axes[1].set_xlabel('Word count')

axes[2].hist(df['avg_option_len'], bins=20, color='#C44E52', edgecolor='white')
axes[2].set_title('Avg Option Length (words)')
axes[2].set_xlabel('Word count')

plt.tight_layout()
plt.show()

## 5. Question-Type Breakdown (Wh-Bucketing)

We classify each question into a Wh-bucket (what, who, where, when, why, how, other) using the rule-based classifier from `preprocessing.py`.

In [ ]:
df['q_type'] = df['question'].apply(label_question_type)
qtype_counts = df['q_type'].value_counts()

fig = px.pie(
    names=qtype_counts.index,
    values=qtype_counts.values,
    title='Question Type Distribution (Wh-Bucketing)',
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.show()

print('Question type counts:')
print(qtype_counts)

## 6. Top Frequent Terms

Most common words across all articles (after lowercasing, excluding very short tokens).

In [ ]:
STOPWORDS = {
    'the','a','an','of','and','or','but','in','on','at','to','for','with',
    'by','from','as','is','are','was','were','be','been','this','that',
    'these','those','it','its','they','them','their','i','you','he','she',
    'we','us','our','your','my','me','his','her','him','have','has','had',
    'do','does','did','not','no','so','if','then','than','would','could',
    'should','will','can','may','might','shall','about','up','out','also',
    'very','more','most','some','any','all','just','how','what','when',
    'where','which','who','whom','why','there','here','each','every',
    'both','few','many','much','own','other','another','such','only',
    'same','too','into','over','after','before','between','through',
    'during','because','while','until','since','said','one','two',
}

all_tokens = []
for article in df['article'].dropna():
    toks = [t for t in tokenize(article) if len(t) > 2 and t not in STOPWORDS]
    all_tokens.extend(toks)

top_30 = Counter(all_tokens).most_common(30)
words, counts = zip(*top_30) if top_30 else ([], [])

fig = px.bar(
    x=list(counts), y=list(words), orientation='h',
    labels={'x': 'Frequency', 'y': 'Word'},
    title='Top 30 Most Frequent Content Words in Articles',
    color_discrete_sequence=['#4C72B0'],
)
fig.update_layout(yaxis=dict(autorange='reversed'))
fig.show()

## 7. Lexical Overlap Analysis

We compute the **Jaccard overlap** between:
- Article tokens vs. gold answer tokens
- Article tokens vs. distractor (wrong answer) tokens
- Question tokens vs. gold answer tokens

This reveals how much word-level signal exists for the verifier.

In [ ]:
def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

overlap_article_gold = []
overlap_article_distractor = []
overlap_question_gold = []

for _, row in df.iterrows():
    a_tok = tokenize(row['article'])
    q_tok = tokenize(row['question'])
    gold_letter = str(row['answer']).strip().upper()
    if gold_letter not in OPTION_LETTERS:
        continue
    gold_tok = tokenize(row[gold_letter])
    
    overlap_article_gold.append(jaccard(a_tok, gold_tok))
    overlap_question_gold.append(jaccard(q_tok, gold_tok))
    
    dist_overlaps = []
    for letter in OPTION_LETTERS:
        if letter != gold_letter:
            d_tok = tokenize(row[letter])
            dist_overlaps.append(jaccard(a_tok, d_tok))
    overlap_article_distractor.append(np.mean(dist_overlaps) if dist_overlaps else 0.0)

overlap_df = pd.DataFrame({
    'Article vs Gold':       overlap_article_gold,
    'Article vs Distractor': overlap_article_distractor,
    'Question vs Gold':      overlap_question_gold,
})

print('Mean Jaccard Overlap:')
print(overlap_df.mean().round(4))
print()

fig = px.box(
    overlap_df.melt(var_name='Pair', value_name='Jaccard'),
    x='Pair', y='Jaccard',
    title='Jaccard Overlap Distribution',
    color='Pair',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

## 8. Sentence Count per Article

In [ ]:
df['n_sentences'] = df['article'].apply(lambda x: len(split_sentences(x)))

fig = px.histogram(
    df, x='n_sentences', nbins=25,
    title='Number of Sentences per Article',
    labels={'n_sentences': 'Sentence Count'},
    color_discrete_sequence=['#DD8452'],
)
fig.show()

print(f'Mean sentences per article: {df["n_sentences"].mean():.1f}')
print(f'Median: {df["n_sentences"].median():.0f}')

## 9. Summary

| Finding | Value |
|---------|-------|
| Total questions | See above |
| Answer class balance | Near uniform (A/B/C/D) |
| Avg article length | See Section 4 |
| Dominant question type | Typically "what" |
| Article-gold overlap > Article-distractor overlap | Confirms lexical signal exists for the verifier |

These findings motivate:
- **One-Hot / TF-IDF** representations (bag-of-words captures the lexical overlap signal)
- **Handcrafted Jaccard/cosine features** (overlap difference between gold and distractor is a learnable signal)
- **Wh-bucketing** for question-type classification (clear category structure)